### Persistent Memory with SQLite

In [1]:
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank
from langchain_community.chat_message_histories import SQLChatMessageHistory
from flashrank import Ranker
from dotenv import load_dotenv
from pathlib import Path
import sqlite3

In [2]:
load_dotenv()

True

Load the vector store and build the reranked retriever


In [3]:
# Load domain-tagged store
persist_dir = r'C:\Users\USER\rag_course\chroma_db_domain'

embeddings = OpenAIEmbeddings()
vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings
)

# Base retriever — pull 20 candidates for reranking
base_retriever = vectorstore.as_retriever(search_kwargs={'k': 20})

# FlashRank reranker - keep best 5
ranker = Ranker(model_name='ms-marco-MiniLM-L-12-v2')
reranker = FlashrankRerank(model='ms-marco-MiniLM-L-12-v2', top_n=5)

reranking_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)

print(f'Reranked retriever ready')
print(f'    Store has {vectorstore._collection.count()} chunks')
print(f'    Flow: 20 candidates -> rerank -> top 5')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Reranked retriever ready
    Store has 80 chunks
    Flow: 20 candidates -> rerank -> top 5


Build the LLM, prompts, and chains

In [4]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Prompt 1: Rewrite follow-up questions into standalone form
rewrite_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You rewrite a user\'s latest question into a standalone question using the chat history.\n'
     '\n'
     'Rules:\n'
     '1. If the latest question refers to something from the history (like "the first one", '
     '"that", "it", "the second option"), REPLACE that reference with the actual item from the history.\n'
     '2. Do NOT answer the question. Only rewrite it.\n'
     '3. If the question is already standalone, return it unchanged.\n'
     '\n'
     'Examples:\n'
     '- History mentions "Malaria" first, then "HIV/AIDS".\n'
     '  User: "Tell me more about the first one."\n'
     '  Rewritten: "Tell me more about malaria."\n'
     '\n'
     '- History mentions "Cassava Mosaic Disease" first, then "Maize Smut".\n'
     '  User: "How do I control the second one?"\n'
     '  Rewritten: "How do I control Maize Smut?"'),
    MessagesPlaceholder('chat_history'),
    ('human', '{input}'),
])

# Prompt 2: Answer using retrieved context
qa_prompt = ChatPromptTemplate.from_messages([
  ('system', 'You are a helpful assistant. '
               'Answer the question using only the provided context. '
               'If you don\'t know, say you don\'t know.'),
    ('human', 'Context:\n{context}\n\nQuestion: {input}'),  
])

# Reusable chains
rewrite_chain = rewrite_prompt | llm | StrOutputParser()
qa_chain = qa_prompt | llm | StrOutputParser()

print('Prompts and chains ready')

Prompts and chains ready


Set up persistent SQLite storage

In [5]:
# Path to the database file

DB_PATH = Path(r'C:\Users\USER\rag_course\chat_history.db')
DB_URL = 'sqlite:///' + DB_PATH.as_posix()    # .as_posix() converts \ to /

print(f'DB path: {DB_PATH}')
print(f'DB URL: {DB_URL}')
print(f'Exists?: {DB_PATH.exists()}')
print('Size:   ', DB_PATH.stat().st_size if DB_PATH.exists() else 0, 'bytes')

DB path: C:\Users\USER\rag_course\chat_history.db
DB URL: sqlite:///C:/Users/USER/rag_course/chat_history.db
Exists?: True
Size:    8192 bytes


In [6]:
conn = sqlite3.connect(DB_PATH)


cursor = conn.cursor()

# List all tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

tables = cursor.fetchall()
print(f'Tables: {tables}')

# If a 'message_store' table exists, show its contents
if ('message_store',) in tables:
    cursor.execute('SELECT * FROM message_store')
    rows = cursor.fetchall()
    print(f'\n{len(rows)} row(s) in message_store:')
    
    for row in rows:
        print(' ', row)
        
conn.close()

Tables: [('message_store',)]

6 row(s) in message_store:
  (1, 'persist-test-1', '{"type": "human", "data": {"content": "What are the top causes of death in Nigeria?", "additional_kwargs": {}, "response_metadata": {}, "type": "human", "name": null, "id": null, "example": false}}')
  (2, 'persist-test-1', '{"type": "ai", "data": {"content": "The top causes of death in Nigeria are:\\n\\n1. Malaria\\n2. Lower Respiratory Infections\\n3. HIV/AIDS\\n4. Diarrheal Diseases\\n5. Road Injuries\\n6. Protein-energy malnutrition\\n7. Cancer\\n8. Meningitis\\n9. Stroke\\n10. Tuberculosis", "additional_kwargs": {}, "response_metadata": {}, "type": "ai", "name": null, "id": null, "example": false, "tool_calls": [], "invalid_tool_calls": [], "usage_metadata": null}}')
  (3, 'persist-test-1', '{"type": "human", "data": {"content": "What are the top causes of death in Nigeria?", "additional_kwargs": {}, "response_metadata": {}, "type": "human", "name": null, "id": null, "example": false}}')
  (4, 'persi

In [7]:
from langchain_community.chat_message_histories import SQLChatMessageHistory
from pathlib import Path

DB_URL = 'sqlite:///C:/Users/USER/rag_course/chat_history.db'   # forward slashes

# This triggers CREATE TABLE IF NOT EXISTS
h = SQLChatMessageHistory(session_id='probe', connection=DB_URL)
h.add_user_message('hello')                                     # writes one row

p = Path(r'C:\Users\USER\rag_course\chat_history.db')
print('File exists?', p.exists())
print('Size:', p.stat().st_size if p.exists() else 0, 'bytes')

File exists? True
Size: 8192 bytes


In [8]:
DB_PATH = Path(r'C:\Users\USER\rag_course\chat_history.db')

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("DELETE FROM message_store WHERE session_id = 'probe'")
conn.commit()

print('Deleted probe session. Rows remaining:', cursor.execute('SELECT COUNT(*) FROM message_store').fetchone()[0])
conn.close()

Deleted probe session. Rows remaining: 6


Define the persistent ask() function

In [9]:
def get_history(session_id):
    '''Return a SQLite-backed chat history for this session.'''
    
    return SQLChatMessageHistory(
        session_id=session_id,
        connection=DB_URL,
    )
    
def ask(question, session_id):
    '''Conversational RAG with persistent history.'''
    
    # Loaded from SQLite
    history = get_history(session_id)
    
    # 1. Fetch messages 
    history_messages = history.messages    # # List of messages
    
    # 2. Rewrite the question if history exists
    if history_messages:
        rewritten = rewrite_chain.invoke({
            'chat_history': history_messages,
            'input': question
        })
        
    else:
        rewritten = question
        
    # 3. Retrieve and rerank
    docs = reranking_retriever.invoke(rewritten)
    context = '\n\n'.join(d.page_content for d in docs)
    
    # 4. Generate the answer
    answer = qa_chain.invoke({
        'context': context,
        'input': rewritten
    })
    
    # 5. Persist both messages to SQLite
    history.add_user_message(question)   # saves user question
    history.add_ai_message(answer)    # saves assistant answer
    
    return answer

print('Persistent ask() ready')

Persistent ask() ready


Have a conversation and save it to SQLite

In [10]:
session_id = 'persist-test-1'

# Turn 1 — health question
q1 = 'What are the top causes of death in Nigeria?'
a1 = ask(q1, session_id)

print('👤 Q1:', q1)
print('🤖 A1:', a1[:400])
print('-' * 70)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


👤 Q1: What are the top causes of death in Nigeria?
🤖 A1: The top causes of death in Nigeria are:

1. Malaria
2. Lower Respiratory Infections
3. HIV/AIDS
4. Diarrheal Diseases
5. Road Injuries
6. Protein-energy malnutrition
7. Cancer
8. Meningitis
9. Stroke
10. Tuberculosis
----------------------------------------------------------------------


In [11]:
# Turn 2 — vague follow-up (same session_id)
q2 = 'Tell me more about the first one.'
a2 = ask(q2, session_id)

print('👤 Q2:', q2)
print('🤖 A2:', a2[:400])
print('-' * 70)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


👤 Q2: Tell me more about the first one.
🤖 A2: Malaria remains the foremost killer disease in Nigeria, with an estimated 300,000 children dying from it each year. It accounts for over 25% of infant mortality (children under one year old), 30% of childhood mortality (children under five years old), and 11% of maternal mortality. At least 50% of the population experiences at least one episode of malaria annually, with children under five having 
----------------------------------------------------------------------


In [12]:
DB_PATH = Path(r'C:\Users\USER\rag_course\chat_history.db')

# Open a connection to the database
conn = sqlite3.connect(DB_PATH)

# Create a cursor to run SQL queries
cursor = conn.cursor()

# Fetch every row, ordered by insertion order
cursor.execute("SELECT id, session_id, message FROM message_store ORDER BY id")

# rows = list of tuples
rows = cursor.fetchall()

print(f'Total rows: {len(rows)}\n')
for row in rows:
    
    # Unpack the three columns
    row_id, sid, msg_json = row
    # Just show a short snippet of the message content
    import json
    try:
        
        # Convert the JSON string back to a dict
        parsed = json.loads(msg_json)
        
        # 'human' or 'ai'
        role = parsed.get('type', '?')
        content = parsed.get('data', {}).get('content', '')[:80]
        
        # Header line
        print(f'[{row_id}] session={sid}  role={role}')
        print(f'     {content}...')
    except Exception as e:
        print(f'[{row_id}] {msg_json[:80]}...')

conn.close()

Total rows: 10

[1] session=persist-test-1  role=human
     What are the top causes of death in Nigeria?...
[2] session=persist-test-1  role=ai
     The top causes of death in Nigeria are:

1. Malaria
2. Lower Respiratory Infecti...
[3] session=persist-test-1  role=human
     What are the top causes of death in Nigeria?...
[4] session=persist-test-1  role=ai
     The top causes of death in Nigeria are:

1. Malaria
2. Lower Respiratory Infecti...
[5] session=persist-test-1  role=human
     Tell me more about the first one....
[6] session=persist-test-1  role=ai
     Malaria remains the foremost killer disease in Nigeria, with an estimated 300,00...
[7] session=persist-test-1  role=human
     What are the top causes of death in Nigeria?...
[8] session=persist-test-1  role=ai
     The top causes of death in Nigeria are:

1. Malaria
2. Lower Respiratory Infecti...
[9] session=persist-test-1  role=human
     Tell me more about the first one....
[10] session=persist-test-1  role=ai
     Mala

In [13]:
import sqlite3                                          # Built-in SQLite
from pathlib import Path                                # Clean paths

DB_PATH = Path(r'C:\Users\USER\rag_course\chat_history.db')   # DB file location

conn = sqlite3.connect(DB_PATH)                         # Open connection
cursor = conn.cursor()                                  # SQL cursor

# Count how many messages are in this session right now
cursor.execute(
    "SELECT COUNT(*) FROM message_store WHERE session_id = ?",
    ('persist-test-1',)
)
count_before = cursor.fetchone()[0]
print(f'Messages for persist-test-1 BEFORE restart: {count_before}')

conn.close()                                            # Close connection

Messages for persist-test-1 BEFORE restart: 10


In [14]:
# After kernel restart — check if history is still there
history = get_history('persist-test-1')

print(f'Messages loaded from SQLite: {len(history.messages)}\n')

for i, msg in enumerate(history.messages):
    role = '👤 User' if msg.type == 'human' else '🤖 Assistant'
    print(f'[{i}] {role}: {msg.content[:100]}')
    print()

Messages loaded from SQLite: 10

[0] 👤 User: What are the top causes of death in Nigeria?

[1] 🤖 Assistant: The top causes of death in Nigeria are:

1. Malaria
2. Lower Respiratory Infections
3. HIV/AIDS
4. D

[2] 👤 User: What are the top causes of death in Nigeria?

[3] 🤖 Assistant: The top causes of death in Nigeria are:

1. Malaria
2. Lower Respiratory Infections
3. HIV/AIDS
4. D

[4] 👤 User: Tell me more about the first one.

[5] 🤖 Assistant: Malaria remains the foremost killer disease in Nigeria, with an estimated 300,000 children dying fro

[6] 👤 User: What are the top causes of death in Nigeria?

[7] 🤖 Assistant: The top causes of death in Nigeria are:

1. Malaria
2. Lower Respiratory Infections
3. HIV/AIDS
4. D

[8] 👤 User: Tell me more about the first one.

[9] 🤖 Assistant: Malaria remains the foremost killer disease in Nigeria, with an estimated 300,000 children dying fro



In [15]:
# Follow-up after restart — uses the restored history
q3 = 'What about the second one?'
a3 = ask(q3, 'persist-test-1')

print('👤 Q3:', q3)
print('🤖 A3:', a3[:400])

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


👤 Q3: What about the second one?
🤖 A3: Lower respiratory infections account for 19% of the top causes of death in Nigeria. They are one of the major health problems in the country, contributing significantly to morbidity and mortality, particularly among children.
